In [18]:
# conda activate ESE527
# Import the library
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os
from pprint import pprint


In [19]:
# Read the Note Event
note_event_df = pd.read_csv('Data/NOTEEVENTS.csv', low_memory=False)
#note_event_df.head()

# Patient selection:
# The ADMISSION.csv contains the admittime dischtime and deathtime
admission_df = pd.read_csv('Data/ADMISSIONS.csv')

print(len(admission_df['SUBJECT_ID'].unique()))
# We should remove the newborn that can not be considered as the new patient
without_NewBorn_df = admission_df[admission_df['DIAGNOSIS'] != 'NEWBORN']
print(f"After remove the newborn case, we still have {len(without_NewBorn_df)} cases")


46520

After remove the newborn case, we still have 51153 cases

In [20]:
#print(len(admission_df['SUBJECT_ID'].unique()))
note_event_df['TEXT'][0]

'Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]\n\n\nService:\nADDENDUM:\n\nRADIOLOGIC STUDIES:  Radiologic studies also included a chest\nCT, which confirmed cavitary lesions in the left lung apex\nconsistent with infectious process/tuberculosis.  This also\nmoderate-sized left pleural effusion.\n\nHEAD CT:  Head CT showed no intracranial hemorrhage or mass\neffect, but old infarction consistent with past medical\nhistory.\n\nABDOMINAL CT:  Abdominal CT showed lesions of\nT10 and sacrum most likely secondary to osteoporosis. These can\nbe followed by repeat imaging as an outpatient.\n\n\n\n                            [**First Name8 (NamePattern2) **] [**First Name4 (NamePattern1) 1775**] [**Last Name (NamePattern1) **], M.D.  [**MD Number(1) 1776**]\n\nDictated By:[**Hospital 1807**]\nMEDQUIST36\n\nD:  [**2151-8-5**]  12:11\nT:  [**2151-8-5**]  12:21\nJOB#:  [**Job Number 1808**]\n'

#### Condition 1: We only consider the patient who stays in the hospital over 30 days (without consider admitted several times)

In [8]:

# Get the HADM_ID
HADM_ID = without_NewBorn_df['HADM_ID']
# Time format 
FMT = "%Y/%m/%d %H:%M" 
over_30Days_ID = []
# Convert ADMITTIME and DISCHTIME to datetime format automatically
without_NewBorn_df['ADMITTIME'] = pd.to_datetime(without_NewBorn_df['ADMITTIME'], errors='coerce')
without_NewBorn_df['DISCHTIME'] = pd.to_datetime(without_NewBorn_df['DISCHTIME'], errors='coerce')

# Iterate all of the admission id to extract which patients stay over 30 days
for hid in HADM_ID:
    admit_time_series = without_NewBorn_df.loc[without_NewBorn_df['HADM_ID'] == hid, 'ADMITTIME']
    discharge_time_series = without_NewBorn_df.loc[without_NewBorn_df['HADM_ID'] == hid, 'DISCHTIME']
    admit_time = admit_time_series.iloc[0]
    discharge_time = discharge_time_series.iloc[0]
    if (discharge_time - admit_time).days >= 30 :
        over_30Days_ID.append(hid)


print(f"Length of the over30Days {len(over_30Days_ID)}")
    
# That ideas is wrong. ACCORDING to the paper, they did not filter the data at beginning. 


C:\Users\judyw\AppData\Local\Temp\ipykernel_38868\1304566885.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  without_NewBorn_df['ADMITTIME'] = pd.to_datetime(without_NewBorn_df['ADMITTIME'], errors='coerce')
C:\Users\judyw\AppData\Local\Temp\ipykernel_38868\1304566885.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  without_NewBorn_df['DISCHTIME'] = pd.to_datetime(without_NewBorn_df['DISCHTIME'], errors='coerce')


Length of the over30Days 2255


####  Follow the logitcs of the papers


In [22]:
from medcat.vocab import Vocab
from medcat.cdb import CDB
from medcat.cat import CAT
from medcat.meta_cat import MetaCAT
import textwrap
from rich import print 
import os

In [6]:
# 纳入所有患者 → 实体提取 → 分桶 → 随机分割（95/5）；
# Install the Medcat process
# Check the model required files 


The med_status exists!!!

In [23]:
# Load the vocab model we downloaded 
vocab = Vocab.load("medcat_models/vocab.dat")
# Load the cdb model you downloaded
cdb = CDB.load('medcat_models/cdb.dat') 


c:\Users\judyw\Desktop\ESE527\project\.conda\Lib\site-packages\pydantic\main.py:426: UserWarning: Pydantic serializer warnings:
  Expected `set[str]` but got `dict` with value `{}` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(
which may or may not work. If you experience any compatibility issues, please reinstall MedCAT
or download the compatible model.


In [26]:
# Run the medcat_status model
mc_status = MetaCAT.load("medcat_models/meta_Status")


In [27]:
# Initial the CAT
cat = CAT(cdb=cdb, config=cdb.config, vocab=vocab, meta_cats=[mc_status])


In [28]:
# 原始文本
text = note_event_df['TEXT'][0]

formatted_text = textwrap.fill(text, width=80)
print(formatted_text)
entities = cat.get_entities(text)
print(entities)
# 替换 `entities` 为 [标记] 版本
# highlighted_text = text
# for entity in sorted(entities.values(), key=lambda x: x["start"], reverse=True):
#     start, end = entity["start"], entity["end"]
#     term = text[start:end]
#     highlighted_text = highlighted_text[:start] + f"[{term}]" + highlighted_text[end:]

# print(highlighted_text)


Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]
Service: ADDENDUM:  RADIOLOGIC STUDIES:  Radiologic studies also included a
chest CT, which confirmed cavitary lesions in the left lung apex consistent with
infectious process/tuberculosis.  This also moderate-sized left pleural
effusion.  HEAD CT:  Head CT showed no intracranial hemorrhage or mass effect,
but old infarction consistent with past medical history.  ABDOMINAL CT:
Abdominal CT showed lesions of T10 and sacrum most likely secondary to
osteoporosis. These can be followed by repeat imaging as an outpatient.
[**First Name8 (NamePattern2) **] [**First Name4 (NamePattern1) 1775**] [**Last
Name (NamePattern1) **], M.D.  [**MD Number(1) 1776**]  Dictated By:[**Hospital
1807**] MEDQUIST36  D:  [**2151-8-5**]  12:11 T:  [**2151-8-5**]  12:21 JOB#:
[**Job Number 1808**]

{
    'entities': {
        0: {
            'pretty_name': 'Hospital admission',
            'cui': '32485007',
            'type_ids': ['28321150'],
            'types': ['procedure'],
            'source_value': 'Admission',
            'detected_name': 'admission',
            'acc': 1.0,
            'context_similarity': 1.0,
            'start': 0,
            'end': 9,
            'icd10': [],
            'ontologies': ['SNOMED-CT'],
            'snomed': [],
            'id': 0,
            'meta_anns': {'Status': {'value': 'Other', 'confidence': 0.7512198686599731, 'name': 'Status'}}
        },
        1: {
            'pretty_name': 'Date (attribute)',
            'cui': '410671006',
            'type_ids': ['43039974'],
            'types': ['attribute'],
            'source_value': 'Date',
            'detected_name': 'date',
            'acc': 0.3409057529447155,
            'context_similarity': 0.3409057529447155,
            'start': 10,
            'end': 14,
            'icd10': [],
            'ontologies': ['SNOMED-CT'],
            'snomed': [],
            'id': 1,
            'meta_anns': {'Status': {'value': 'Other', 'confidence': 0.68000727891922, 'name': 'Status'}}
        },
        2: {
            'pretty_name': 'Discharge',
            'cui': '75823008',
            'type_ids': ['33782986'],
            'types': ['morphologic abnormality'],
            'source_value': 'Discharge',
            'detected_name': 'discharge',
            'acc': 0.99,
            'context_similarity': 0.99,
            'start': 39,
            'end': 48,
            'icd10': [],
            'ontologies': ['SNOMED-CT'],
            'snomed': [],
            'id': 2,
            'meta_anns': {'Status': {'value': 'Affirmed', 'confidence': 0.8986527323722839, 'name': 'Status'}}
        },
        3: {
            'pretty_name': 'Dates - food',
            'cui': '227423000',
            'type_ids': ['91187746'],
            'types': ['substance'],
            'source_value': 'Date',
            'detected_name': 'date',
            'acc': 0.6775456875925483,
            'context_similarity': 0.6775456875925483,
            'start': 49,
            'end': 53,
            'icd10': [],
            'ontologies': ['SNOMED-CT'],
            'snomed': [],
            'id': 3,
            'meta_anns': {'Status': {'value': 'Other', 'confidence': 0.68000727891922, 'name': 'Status'}}
        },
        4: {
            'pretty_name': 'Services',
            'cui': '224930009',
            'type_ids': ['7882689'],
            'types': ['qualifier value'],
            'source_value': 'Service',
            'detected_name': 'service',
            'acc': 1.0,
            'context_similarity': 1.0,
            'start': 73,
            'end': 80,
            'icd10': [],
            'ontologies': ['SNOMED-CT'],
            'snomed': [],
            'id': 4,
            'meta_anns': {'Status': {'value': 'Affirmed', 'confidence': 0.8120161294937134, 'name': 'Status'}}
        },
        5: {
            'pretty_name': 'Radiologic',
            'cui': '5526005',
            'type_ids': ['7882689'],
            'types': ['qualifier value'],
            'source_value': 'RADIOLOGIC',
            'detected_name': 'radiologic',
            'acc': 1.0,
            'context_similarity': 1.0,
            'start': 93,
            'end': 103,
            'icd10': [],
            'ontologies': ['SNOMED-CT'],
            'snomed': [],
            'id': 5,
            'meta_anns': {'Status': {'value': 'Affirmed', 'confidence': 0.9902357459068298, 'name': 'Status'}}
        },
        6: {
            'pretty_name': 'Study',
            'cui': '224699009',
            'type_ids': ['75168589'],
            'types': ['environment'],
            'source_value': 'STUDIES',
            'detected_name': 'study',
            'acc': 1.0,
            'context_similarity': 1.0,
            'start': 104,
            'end': 111,
            'icd10': [],
            'on